# Proyecto final
##### Simulación de Procesos Financieros
##### María Fernanda Gómez Flores, Abraham Flores y Jimena Muñoz

En este proyecto vamos a simular un portafolio de cinco acciones durante un año. Queremos comparar los métodos de simulación que vimos durante el semestre y ver qué pasa con el riesgo del portafolio cuando cambian las condiciones del mercado.

**Datos:** usamos precios de cierre de Yahoo Finance. Los resultados se obtienen al ejecutar las celdas en orden.

### 1. Descargar datos

Elegimos AMD (tecnología), DAL (aerolíneas), COST (comercio), NU (servicios financieros) y CVX (energía). Escogimos empresas de diferentes sectores para ver si al repartir el dinero entre ellas podemos reducir el riesgo.

Usamos tres años de precios de cierre en dólares, del 21 de noviembre de 2023 al 20 de noviembre de 2026. No incluimos dividendos.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm, qmc

In [ ]:
tickers = ['AMD', 'DAL', 'COST', 'NU', 'CVX']
prices = yf.download(tickers, start='2023-11-21', end='2026-11-21', auto_adjust=False)['Close']
prices = prices[tickers].dropna()
prices

### 2. Rendimientos logarítmicos

Primero sacamos los rendimientos diarios de cada acción para poder comparar sus movimientos sin importar cuánto cuesta cada una.

In [ ]:
rets = np.log(prices / prices.shift(1)).dropna()
rets

In [ ]:
prices.plot(figsize=(10,5))
plt.title("Precios históricos")
plt.xlabel("Fecha")
plt.ylabel("Precio (USD)")
plt.show()

In [ ]:
rets.hist(bins=50, figsize=(10,7))
plt.show()

### 3. Rendimiento, volatilidad y correlación
Como los rendimientos son diarios, multiplicamos la media por 252 y la volatilidad por la raíz de 252 para tener los datos anualizados.

In [ ]:
mu = rets.mean() * 252
vol = rets.std() * np.sqrt(252)

print("Rendimiento anual (%):")
print((mu*100).round(2))
print("Volatilidad anual (%):")
print((vol*100).round(2))

In [ ]:
R = rets.corr()
R.round(3)

**Lo que observamos:** comparamos el rendimiento y la volatilidad de las cinco acciones. La acción con mayor rendimiento histórico no necesariamente es la que más nos conviene para reducir el riesgo. En la matriz de correlación podemos ver qué acciones se mueven más parecido. Aunque sean de sectores distintos, no podemos considerar que todos sus movimientos sean independientes.

### 4. Covarianza y Cholesky

Calculamos la matriz de covarianzas anualizada y aplicamos Cholesky para tomar en cuenta la relación entre los rendimientos de las cinco acciones.

In [ ]:
Sigma = rets.cov()*252
L = np.linalg.cholesky(Sigma)
print(Sigma)
L

### 5. Monte Carlo

Usamos 10,000 simulaciones y un horizonte de un año. Generamos normales independientes y después las multiplicamos por L para obtener Z. Como Z ya contiene la volatilidad y mu es la media anual de los rendimientos logarítmicos, no volvemos a multiplicar por la volatilidad ni a restar la mitad de la varianza.

In [ ]:
T = 1
M = 10000
V0 = 100000
S0 = prices.iloc[-1].values
w = np.array([0.20, 0.20, 0.20, 0.20, 0.20])

np.random.seed(67)
epsilon = np.random.normal(0, 1, (len(tickers), M))
Z = L @ epsilon
Z.T

In [ ]:
ST = S0 * np.exp(mu.values*T + np.sqrt(T)*Z.T)
ST = pd.DataFrame(ST, columns=tickers)
ST.head()

### 6. Valor final del portafolio
Invertimos 20% en cada acción y usamos acciones fraccionarias para invertir exactamente los $100,000 USD. Al valor final le restamos el capital inicial para obtener las ganancias y pérdidas.

In [ ]:
N = V0*w/S0
print(pd.DataFrame({"Acción": tickers, "Cantidad": N, "Peso": w}))

VT = V0 * np.sum((ST.values / S0) * w, axis=1)
PnL = VT - V0
rend_port = PnL / V0

print(f"Valor final promedio: ${np.mean(VT):,.2f}")
print(f"Ganancia promedio: ${np.mean(PnL):,.2f}")
print("Rendimiento promedio (%):", round(np.mean(rend_port)*100,2))
print("Probabilidad de pérdida (%):", round(np.mean(PnL<0)*100,2))

In [ ]:
plt.hist(VT, bins=50)
plt.axvline(V0, color="black", linestyle="--", label="Capital inicial")
plt.axvline(np.mean(VT), color="red", linestyle="--", label="Media")
plt.title("Valor final del portafolio")
plt.xlabel("Valor final (USD)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

In [ ]:
plt.hist(rend_port*100, bins=50)
plt.axvline(0, color="black", linestyle="--")
plt.title("Rendimientos del portafolio")
plt.xlabel("Rendimiento (%)")
plt.ylabel("Frecuencia")
plt.show()

### 7. VaR y CVaR

Sacamos el percentil 5 de las ganancias y pérdidas. El VaR es ese percentil con signo contrario y el CVaR es la pérdida promedio dentro del 5% de los escenarios más bajos.

In [ ]:
q = np.percentile(PnL, 5)
VaR = -q
CVaR = -np.mean(PnL[PnL <= q])
error_MC = np.std(VT, ddof=1)/np.sqrt(M)

print("VaR 95%:", round(VaR,2))
print("CVaR 95%:", round(CVaR,2))
print("Error estándar:", round(error_MC,2))

In [ ]:
plt.hist(PnL, bins=50)
plt.axvline(q, color="red", linestyle="--", label="VaR 95%")
plt.axvline(-CVaR, color="orange", linestyle="--", label="CVaR 95%")
plt.axvline(0, color="black", linestyle="--")
plt.xlabel("Ganancia o pérdida (USD)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### 8. Variables antitéticas

Tomamos 5,000 valores de Z y después hacemos lo mismo con -Z para tener 10,000 escenarios. Sacamos el promedio de cada pareja para estimar el valor del portafolio. Para el VaR y CVaR usamos los 10,000 valores individuales porque queremos medir las posibles pérdidas del portafolio, no las pérdidas del promedio de dos escenarios.

In [ ]:
mitad = M//2
Z_ant = Z[:, :mitad]

ST_z = S0*np.exp(mu.values*T + np.sqrt(T)*Z_ant.T)
ST_menosZ = S0*np.exp(mu.values*T - np.sqrt(T)*Z_ant.T)

VT_z = V0*np.sum((ST_z/S0)*w, axis=1)
VT_menosZ = V0*np.sum((ST_menosZ/S0)*w, axis=1)

V_promedio = (VT_z + VT_menosZ)/2
valor_ant = np.mean(V_promedio)
error_ant = np.std(V_promedio, ddof=1)/np.sqrt(mitad)

VT_ant = np.concatenate((VT_z, VT_menosZ))
PnL_ant = VT_ant - V0
q_ant = np.percentile(PnL_ant, 5)
VaR_ant = -q_ant
CVaR_ant = -np.mean(PnL_ant[PnL_ant <= q_ant])

print("Valor esperado:", round(valor_ant,2))
print("Error estándar:", round(error_ant,2))

### 9. Variable de control

Tomamos Y como el valor final del portafolio y X como el precio final de AMD. Escogimos AMD porque forma parte del portafolio y su precio está relacionado con Y. Calculamos EX con la esperanza de la distribución lognormal y después sacamos b para ajustar los valores de Y.

In [ ]:
Y = VT
X = ST["AMD"].values
EX = S0[0]*np.exp(mu.iloc[0]*T + 0.5*vol.iloc[0]**2*T)

x_media = np.mean(X)
y_media = np.mean(Y)

b = np.sum((X-x_media)*(Y-y_media))/np.sum((X-x_media)**2)
Y_control = Y-b*(X-EX)
valor_control = np.mean(Y_control)
error_control = np.std(Y_control, ddof=1)/np.sqrt(M)

print("Sin variable de control:", round(y_media,2))
print("Con variable de control:", round(valor_control,2))
print("Reducción de varianza (%):", round((1-np.var(Y_control,ddof=1)/np.var(Y,ddof=1))*100,2))

Aquí estamos ajustando la estimación del valor promedio, no la distribución de precios finales. Por eso reportamos el VaR y CVaR del Monte Carlo original también en esta fila; no son estimaciones independientes con variable de control.

In [ ]:
VaR_control = VaR
CVaR_control = CVaR

### 10. Quasi Monte Carlo (Sobol)

Primero generamos uniformes con Sobol y después los pasamos a una normal con norm.ppf. Usamos cinco dimensiones, una por acción. Como Sobol trabaja mejor con potencias de 2, generamos 16,384 puntos y usamos los primeros 10,000 para comparar con los otros métodos. Al recortar los puntos perdemos parte de las propiedades de balance de la secuencia.

In [ ]:
sobol = qmc.Sobol(d=len(tickers), scramble=True, seed=32)
u = sobol.random_base2(m=14)
u = u[:M]
z = norm.ppf(np.clip(u, 1e-12, 1-1e-12))

Z_sobol = L @ z.T
ST_sobol = S0*np.exp(mu.values*T + np.sqrt(T)*Z_sobol.T)
VT_sobol = V0*np.sum((ST_sobol/S0)*w, axis=1)
PnL_sobol = VT_sobol - V0

valor_sobol = np.mean(VT_sobol)
q_sobol = np.percentile(PnL_sobol,5)
VaR_sobol = -q_sobol
CVaR_sobol = -np.mean(PnL_sobol[PnL_sobol <= q_sobol])

Para el error estándar repetimos Sobol con 10 semillas distintas. Como sus puntos no son independientes, usamos la desviación de los 10 valores estimados para aproximar el error de una corrida de 10,000 escenarios. No dividimos entre la raíz de 10 porque en la tabla estamos reportando una sola corrida, no el promedio de las 10.

In [ ]:
estimaciones = []

for i in range(10):
    sobol = qmc.Sobol(d=len(tickers), scramble=True, seed=i)
    u = sobol.random_base2(m=14)[:M]
    z = norm.ppf(np.clip(u, 1e-12, 1-1e-12))
    Z_rep = L @ z.T
    ST_rep = S0*np.exp(mu.values*T + np.sqrt(T)*Z_rep.T)
    VT_rep = V0*np.sum((ST_rep/S0)*w, axis=1)
    estimaciones.append(np.mean(VT_rep))

error_sobol = np.std(estimaciones, ddof=1)
print("Valor estimado:", round(valor_sobol,2))
print("Error estándar aproximado:", round(error_sobol,2))

### 11. Comparación de métodos

In [ ]:
tabla = pd.DataFrame({
    "Método": ["Monte Carlo", "Variables antitéticas", "Variable de control", "Quasi Monte Carlo"],
    "Valor estimado": [np.mean(VT), valor_ant, valor_control, valor_sobol],
    "VaR95": [VaR, VaR_ant, VaR_control, VaR_sobol],
    "CVaR95": [CVaR, CVaR_ant, CVaR_control, CVaR_sobol],
    "Error estándar": [error_MC, error_ant, error_control, error_sobol]
})
tabla.round(2)

**Lo que observamos:** los cuatro valores estimados se pueden comparar porque usamos los mismos parámetros y el mismo número de escenarios. Monte Carlo es el más sencillo, pero las antitéticas pueden reducir la variación al usar Z y -Z. Para la variable de control necesitamos conocer EX y que X esté relacionada con Y. Sobol reparte mejor los puntos, aunque al usar 10,000 en lugar de una potencia de 2 perdemos parte de esa ventaja. En la siguiente celda revisamos cuál tiene menor error estándar. Esto no significa que también tenga el VaR más preciso.

In [ ]:
print("Método con menor error estándar:", tabla.loc[tabla["Error estándar"].idxmin(),"Método"])

### 12. Stress testing
Vamos a usar las mismas normales que en el escenario base para comparar qué pasa cuando cambiamos una condición a la vez.

**Escenario 1. Caída general del mercado**
Suponemos una caída adicional de 35% en los precios finales de todas las acciones.

In [ ]:
ST_1 = ST.values*(1-0.35)
VT_1 = V0*np.sum((ST_1/S0)*w, axis=1)
PnL_1 = VT_1-V0

**Escenario 2. Doble volatilidad**
Multiplicamos Z por 2 porque ya contiene la volatilidad de cada acción. Dejamos fija la media logarítmica, por lo que el valor promedio puede subir por la forma exponencial del modelo, aunque el riesgo aumente.

In [ ]:
ST_2 = S0*np.exp(mu.values*T + 2*np.sqrt(T)*Z.T)
VT_2 = V0*np.sum((ST_2/S0)*w, axis=1)
PnL_2 = VT_2-V0

**Escenario 3. Colapso de diversificación**
Ponemos una correlación de 0.99 entre las acciones y dejamos 1 en la diagonal. Volvemos a calcular la covarianza y Cholesky, sin cambiar las volatilidades individuales.

In [ ]:
R_3 = np.ones((len(tickers),len(tickers)))*0.99

for i in range(len(tickers)):
    R_3[i,i] = 1

Sigma_3 = np.zeros((len(tickers),len(tickers)))
for i in range(len(tickers)):
    for j in range(len(tickers)):
        Sigma_3[i,j] = R_3[i,j]*vol.iloc[i]*vol.iloc[j]

L_3 = np.linalg.cholesky(Sigma_3)
Z_3 = L_3 @ epsilon
ST_3 = S0*np.exp(mu.values*T + np.sqrt(T)*Z_3.T)
VT_3 = V0*np.sum((ST_3/S0)*w, axis=1)
PnL_3 = VT_3-V0

In [ ]:
escenarios = {"Base": PnL, "Caída 35%": PnL_1, "Doble volatilidad": PnL_2, "Correlación 0.99": PnL_3}
resumen = []

for nombre, pnl in escenarios.items():
    q = np.percentile(pnl,5)
    resumen.append({"Escenario": nombre, "VaR95": -q, "CVaR95": -np.mean(pnl[pnl <= q])})

tabla_stress = pd.DataFrame(resumen)
tabla_stress.round(2)

In [ ]:
for nombre, pnl in escenarios.items():
    plt.hist(pnl, bins=50, alpha=0.4, label=nombre)

plt.axvline(0, color="black", linestyle="--")
plt.xlabel("Ganancia o pérdida (USD)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

**Lo que observamos:** en la caída del mercado los precios finales bajan en todos los escenarios. Al duplicar la volatilidad aumenta la dispersión y al subir las correlaciones perdemos parte del beneficio de diversificar. Comparamos el VaR y CVaR de cada caso con el escenario base para ver cuál nos afecta más.

### 13. Extra - Optimización de pesos
Planteamos un cliente que busca reducir el riesgo. Usamos como función de costo la varianza anualizada del portafolio, $w^T\Sigma w$. Vamos a probar pesos de 5% en 5%, con un mínimo de 5% y un máximo de 40% por acción, sin ventas en corto. La suma debe ser 100%. Así encontramos la mejor combinación de las que probamos, no necesariamente la mejor entre todos los pesos posibles.

In [ ]:
mejor_costo = w @ Sigma.values @ w
w_opt = w.copy()

for a in range(1,9):
    for b in range(1,9):
        for c in range(1,9):
            for d in range(1,9):
                e = 20-a-b-c-d
                if 1 <= e <= 8:
                    w_prueba = np.array([a,b,c,d,e])/20
                    costo = w_prueba @ Sigma.values @ w_prueba
                    if costo < mejor_costo:
                        mejor_costo = costo
                        w_opt = w_prueba.copy()

pd.DataFrame({"Acción": tickers, "Peso original (%)": w*100, "Peso optimizado (%)": w_opt*100})

In [ ]:
VT_opt = V0*np.sum((ST.values/S0)*w_opt, axis=1)
PnL_opt = VT_opt-V0
q_opt = np.percentile(PnL_opt,5)
VaR_opt = -q_opt
CVaR_opt = -np.mean(PnL_opt[PnL_opt <= q_opt])

vol_original = np.sqrt(w @ Sigma.values @ w)
vol_opt = np.sqrt(mejor_costo)

tabla_opt = pd.DataFrame({
    "Portafolio": ["Original", "Optimizado"],
    "Valor esperado": [np.mean(VT), np.mean(VT_opt)],
    "Volatilidad histórica (%)": [vol_original*100, vol_opt*100],
    "VaR95": [VaR, VaR_opt],
    "CVaR95": [CVaR, CVaR_opt]
})
tabla_opt.round(2)

In [ ]:
plt.hist(PnL, bins=50, alpha=0.4, label="Original")
plt.hist(PnL_opt, bins=50, alpha=0.4, label="Optimizado")
plt.axvline(0, color="black", linestyle="--")
plt.xlabel("Ganancia o pérdida (USD)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

**Lo que observamos:** la combinación de pesos que encontramos busca reducir la volatilidad histórica. Comparamos también el valor esperado, el VaR y el CVaR para ver cuánto rendimiento estamos dejando de obtener y cómo cambia el riesgo. Los pesos corresponden a la mejor combinación de las que probamos de 5% en 5%.

### 14. Conclusiones

**¿Invertiríamos en este portafolio?**

Para el cliente que planteamos preferiríamos revisar el portafolio optimizado antes de invertir en el original. No basta con ver el rendimiento esperado, también necesitamos considerar las posibles pérdidas y qué tanto se concentra el dinero en pocas acciones.

**¿Cuál fue el principal factor de riesgo?**

Comparamos los tres escenarios con el caso base. La caída general del mercado reduce directamente el valor final, mientras que el aumento de correlación muestra que diversificar entre sectores no nos protege completamente cuando los activos se mueven juntos. Los resultados de la comparación se muestran abajo.

**¿Qué limitación importante tiene el modelo?**

Usamos datos históricos y suponemos que las medias, volatilidades y correlaciones se mantienen durante el siguiente año. En la realidad pueden cambiar y los rendimientos logarítmicos no siempre se comportan como una normal. Tampoco consideramos dividendos ni costos de transacción. Por eso el rendimiento estimado no es un rendimiento garantizado.

In [ ]:
print("Valor esperado original:", round(np.mean(VT),2))
print("Rendimiento esperado (%):", round(np.mean(rend_port)*100,2))
print("VaR original:", round(VaR,2))
print("CVaR original:", round(CVaR,2))
print("Volatilidad original (%):", round(vol_original*100,2))
print("Volatilidad optimizada (%):", round(vol_opt*100,2))
print("CVaR optimizado:", round(CVaR_opt,2))
print("Escenario con mayor CVaR:", tabla_stress.loc[tabla_stress["CVaR95"].idxmax(),"Escenario"])
print("Método con menor error estándar:", tabla.loc[tabla["Error estándar"].idxmin(),"Método"])